# 02 · Caixa **ou** recorte — o que muda quando a máquina desenha o contorno

Demo curta, para emendar na 01. A pergunta de palco é: *"por que existe mais de
um tipo de modelo?"*

- **Detecção** responde *onde está* → caixa retangular
- **Segmentação** responde *qual é exatamente o pixel* → contorno recortado

A caixa serve para contar. O contorno serve para **medir**: área ocupada, quanto
de uma prateleira está cheia, quanto de um terreno é telhado.

In [ ]:
# ── 1. instala a biblioteca e monta o Google Drive ──
%pip install -q ultralytics
from google.colab import drive
drive.mount('/content/drive')

from ultralytics import YOLO
import ultralytics, torch, os, glob
ultralytics.checks()
print("GPU disponivel:", torch.cuda.is_available())

In [ ]:
# ── ajuste de PALCO: tudo grande, porque a sala enxerga de 6 a 10 m ──
import matplotlib
matplotlib.rcParams.update({
    "figure.figsize": (16, 9),
    "figure.dpi": 110,
    "font.size": 22,
    "axes.titlesize": 30,
    "axes.labelsize": 24,
    "xtick.labelsize": 20,
    "ytick.labelsize": 20,
    "legend.fontsize": 22,
    "axes.grid": True,
    "grid.alpha": .25,
    "axes.facecolor": "#0d1117",
    "figure.facecolor": "#0d1117",
    "text.color": "#e6edf3",
    "axes.labelcolor": "#e6edf3",
    "xtick.color": "#e6edf3",
    "ytick.color": "#e6edf3",
    "axes.edgecolor": "#30363d",
    "axes.titlecolor": "#3fe0a8",
})
VERDE, VERMELHO, CINZA = "#3fe0a8", "#ff5c5c", "#7d8590"
DRIVE = "/content/drive/MyDrive/PALESTRA-IA"
print("palco configurado · raiz no Drive:", DRIVE)

In [ ]:
det = YOLO(f"{DRIVE}/00-pesos/yolo11n.pt")
seg = YOLO(f"{DRIVE}/00-pesos/yolo11n-seg.pt")

import glob
entradas = [e for e in sorted(glob.glob(f"{DRIVE}/02-segmentacao/entrada/*"))
            if e.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))]
IMG = entradas[0] if entradas else "https://ultralytics.com/images/bus.jpg"
print("usando:", IMG)

In [ ]:
# ── lado a lado: a mesma cena, dois modelos ──
import matplotlib.pyplot as plt, cv2
rd = det.predict(IMG, conf=.35, verbose=False)[0]
rs = seg.predict(IMG, conf=.35, verbose=False)[0]

fig, axs = plt.subplots(1, 2, figsize=(22, 10))
for ax, r, t in ((axs[0], rd, "DETECÇÃO · onde está"),
                 (axs[1], rs, "SEGMENTAÇÃO · qual pixel é")):
    ax.imshow(cv2.cvtColor(r.plot(line_width=4), cv2.COLOR_BGR2RGB))
    ax.set_title(t, fontsize=30); ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# ── o que so a segmentacao entrega: AREA de cada objeto ──
import numpy as np
if rs.masks is None:
    print("nenhuma mascara nesta imagem")
else:
    total_px = rs.orig_shape[0] * rs.orig_shape[1]
    linhas = []
    for m, c in zip(rs.masks.data.cpu().numpy(), rs.boxes.cls):
        linhas.append((seg.names[int(c)], m.sum() / m.size * 100))
    linhas.sort(key=lambda x: -x[1])
    print(f"{'objeto':<16} {'% da imagem':>12}")
    print("-" * 30)
    for nome, pct in linhas[:10]:
        print(f"{nome:<16} {pct:>11.1f}%")
    print("-" * 30)
    print(f"{'ocupacao total':<16} {sum(p for _, p in linhas):>11.1f}%")

> **Fala de palco:** *"a caixa me diz que tem cinco garrafas. O contorno me diz
> que a prateleira está 38% vazia. São perguntas diferentes — e é por isso que
> escolher o modelo é parte do problema, não detalhe técnico."*